In [1]:
# first try: can we even import things?
import arxiv
import pandas as pd
from datetime import datetime

print("arxiv module:", arxiv)
print("pandas version:", pd.__version__)

arxiv module: <module 'arxiv' from '/opt/anaconda3/envs/arxiv-trend-predictor-001/lib/python3.13/site-packages/arxiv/__init__.py'>
pandas version: 3.0.0


In [2]:
# super naive test: just grab a few cs.AI papers and look at the titles
search = arxiv.Search(
    query="cat:cs.AI",
    max_results=5,
    sort_by=arxiv.SortCriterion.SubmittedDate,
    sort_order=arxiv.SortOrder.Descending,
)

client = arxiv.Client(page_size=5)

results = list(client.results(search))
len(results), [r.title for r in results]


(5,
 ['A Very Big Video Reasoning Suite',
  'Behavior Learning (BL): Learning Hierarchical Optimization Structures from Data',
  'Agentic AI for Scalable and Robust Optical Systems Control',
  'Recurrent Structural Policy Gradient for Partially Observable Mean Field Games',
  'KNIGHT: Knowledge Graph-Driven Multiple-Choice Question Generation with Adaptive Hardness Calibration'])

In [3]:
# now try: restrict by year using the submittedDate range syntax
year = 2024
category = "cs.AI"
start_date = f"{year}0101"
end_date = f"{year}1231"

query = f"cat:{category} AND submittedDate:[{start_date} TO {end_date}]"
print("query =", query)

search_2024 = arxiv.Search(
    query=query,
    max_results=10,
    sort_by=arxiv.SortCriterion.SubmittedDate,
    sort_order=arxiv.SortOrder.Descending,
)

client_2024 = arxiv.Client(page_size=10)
sample_results = list(client_2024.results(search_2024))
len(sample_results)


query = cat:cs.AI AND submittedDate:[20240101 TO 20241231]


10

In [4]:
# quick and dirty look at what one result object has
r0 = sample_results[0]
print("ID:", r0.entry_id)
print("title:", r0.title)
print("published:", r0.published)
print("primary category:", r0.primary_category)
print("all categories:", r0.categories)
print("authors:", [a.name for a in r0.authors])


ID: http://arxiv.org/abs/2501.00669v1
title: Leaf diseases detection using deep learning methods
published: 2024-12-31 22:56:19+00:00
primary category: cs.LG
all categories: ['cs.LG', 'cs.AI', 'cs.CV']
authors: ['El Houcine El Fatimi']


In [5]:
# throw these into a DataFrame just to see the shape
rows = []
for r in sample_results:
    rows.append(
        {
            "arxiv_id": r.entry_id.split("/")[-1],
            "title": r.title.replace("\n", " ").strip(),
            "abstract": r.summary.replace("\n", " ").strip(),
            "published_date": r.published.strftime("%Y-%m-%d"),
            "primary_category": r.primary_category,
            "all_categories": ", ".join(r.categories),
            "authors": ", ".join(a.name for a in r.authors[:10]),
        }
    )

df_preview = pd.DataFrame(rows)
df_preview.head(3)


,arxiv_id,title,abstract,published_date,primary_category,all_categories,authors
0,2501.00669v1,Leaf diseases detection using deep learning me...,"This study, our main topic is to devlop a new ...",2024-12-31,cs.LG,"cs.LG, cs.AI, cs.CV",El Houcine El Fatimi
1,2501.00664v3,Grade Inflation in Generative Models,"Generative models hold great potential, but on...",2024-12-31,cs.AI,"cs.AI, cs.LG, stat.ML","Phuc Nguyen, Miao Li, Alexandra Morgan, Rima A..."
2,2501.00663v1,Titans: Learning to Memorize at Test Time,Over more than a decade there has been an exte...,2024-12-31,cs.LG,"cs.LG, cs.AI, cs.CL","Ali Behrouz, Peilin Zhong, Vahab Mirrokni"


In [6]:
# wrap that into a first rough helper (still kind of ad-hoc)
def fetch_one_year_raw(category: str, year: int, max_results: int = 50) -> pd.DataFrame:
    start_date = f"{year}0101"
    end_date = f"{year}1231"
    query = f"cat:{category} AND submittedDate:[{start_date} TO {end_date}]"

    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.SubmittedDate,
        sort_order=arxiv.SortOrder.Descending,
    )
    client = arxiv.Client(page_size=100)

    rows: list[dict] = []
    for r in client.results(search):
        rows.append(
            {
                "arxiv_id": r.entry_id.split("/")[-1],
                "title": r.title.replace("\n", " ").strip(),
                "abstract": r.summary.replace("\n", " ").strip(),
                "published_date": r.published.strftime("%Y-%m-%d"),
            }
        )

    return pd.DataFrame(rows)

test_df = fetch_one_year_raw("cs.AI", 2022, max_results=20)
len(test_df), test_df.head(2)


(20,
        arxiv_id                                              title  \
 0  2301.00303v1  Rethinking with Retrieval: Faithful Large Lang...   
 1  2301.01148v1  MERLIN: Multi-agent offline and transfer learn...   
 
                                             abstract published_date  
 0  Despite the success of large language models (...     2022-12-31  
 1  The decarbonization of buildings presents new ...     2022-12-31  )

## Now make it more structured / configurable

The cells below are closer to the "real" scraper the app will use, but we keep them here in the same notebook so you can see the evolution.

In [7]:
from __future__ import annotations

import json
import time
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

from backend.core.constants import (
    ARXIV_DELAY_SECONDS,
    ARXIV_NUM_RETRIES,
    TECH_CATEGORIES,
)
from backend.core.config import settings
from backend.logger import get_logger

logger = get_logger(__name__)


In [8]:
# more polished version of the per-category/year fetch
def fetch_papers_for_category_year(
    category: str,
    year: int,
    max_results: int,
) -> List[Dict]:
    """Fetch papers for a specific category and year from arXiv."""
    papers: List[Dict] = []

    start_date = f"{year}0101"
    end_date = f"{year}1231"
    query = f"cat:{category} AND submittedDate:[{start_date} TO {end_date}]"

    try:
        client = arxiv.Client(
            page_size=100,
            delay_seconds=ARXIV_DELAY_SECONDS,
            num_retries=ARXIV_NUM_RETRIES,
        )

        search = arxiv.Search(
            query=query,
            max_results=max_results,
            sort_by=arxiv.SortCriterion.SubmittedDate,
            sort_order=arxiv.SortOrder.Descending,
        )

        for result in client.results(search):
            paper = {
                "arxiv_id": result.entry_id.split("/")[-1],
                "title": result.title.replace("\n", " ").strip(),
                "abstract": result.summary.replace("\n", " ").strip(),
                "published_date": result.published.strftime("%Y-%m-%d"),
                "updated_date": (
                    result.updated.strftime("%Y-%m-%d") if result.updated else None
                ),
                "year": result.published.year,
                "month": result.published.month,
                "primary_category": result.primary_category,
                "all_categories": ", ".join(result.categories),
                "tech_domain": TECH_CATEGORIES.get(category, "Other"),
                "authors": ", ".join(a.name for a in result.authors[:10]),
                "author_count": len(result.authors),
                "doi": result.doi,
                "journal_ref": result.journal_ref,
                "pdf_url": result.pdf_url,
                "title_length": len(result.title.split()),
                "abstract_length": len(result.summary.split()),
            }
            papers.append(paper)
    except Exception as exc:
        logger.error(f"Error fetching {category} for {year}: {exc}")

    return papers


In [9]:
# small sanity check on the structured helper
sample_structured = fetch_papers_for_category_year("cs.AI", 2023, max_results=10)
len(sample_structured), list(sample_structured[0].keys())[:8]


(10,
 ['arxiv_id',
  'title',
  'abstract',
  'published_date',
  'updated_date',
  'year',
  'month',
  'primary_category'])

In [10]:
def collect_all_data(
    categories: Dict[str, str] | None = None,
    start_year: int | None = None,
    end_year: int | None = None,
    papers_per_cat_year: int | None = None,
    save_incremental: bool = False,  # kept for API compatibility
    output_dir: Path | None = None,
) -> pd.DataFrame:
    """Main collection function - iterates through all categories and years."""
    cats = categories or TECH_CATEGORIES
    start = start_year or settings.arxiv_start_year
    end = end_year or settings.arxiv_end_year
    per_year = papers_per_cat_year or settings.papers_per_category_per_year
    out_dir = output_dir or settings.arxiv_data_dir
    out_dir.mkdir(parents=True, exist_ok=True)

    all_papers: List[Dict] = []
    total_categories = len(cats)
    total_years = end - start + 1
    total_iterations = total_categories * total_years
    current_iteration = 0

    logger.info(
        "Starting collection: {} categories × {} years (max ~{} papers)",
        total_categories,
        total_years,
        total_categories * total_years * per_year,
    )

    import time as _time  # local alias just to show the evolution / junk

    for cat_code, cat_name in cats.items():
        for year in range(start, end + 1):
            current_iteration += 1
            progress = (current_iteration / total_iterations) * 100

            logger.info("[{:.1f}%] Fetching {} ({}) - {}", progress, cat_name, cat_code, year)

            papers = fetch_papers_for_category_year(
                category=cat_code,
                year=year,
                max_results=per_year,
            )

            all_papers.extend(papers)
            logger.info("→ Got {} papers (Total: {:,})", len(papers), len(all_papers))

            _time.sleep(1)  # be nice to arXiv

    df = pd.DataFrame(all_papers)

    if df.empty:
        logger.warning("No papers collected.")
        return df

    before = len(df)
    df = df.drop_duplicates(subset=["arxiv_id"], keep="first")
    after = len(df)
    logger.info(
        "Collection complete! Total unique papers: {:,} (removed {:,} duplicates)",
        after,
        before - after,
    )

    return df


In [11]:
def save_data(
    df: pd.DataFrame,
    filename_prefix: str = "arxiv_tech_papers",
    output_dir: Path | None = None,
) -> Tuple[Path, Path, Path]:
    """Save the collected data in multiple formats."""
    out_dir = output_dir or settings.arxiv_data_dir
    out_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    csv_path = out_dir / f"{filename_prefix}_{timestamp}.csv"
    df.to_csv(csv_path, index=False)
    logger.info("Saved CSV: {}", csv_path)

    json_path = out_dir / f"{filename_prefix}_{timestamp}.json"
    df.to_json(json_path, orient="records", indent=2)
    logger.info("Saved JSON: {}", json_path)

    stats = {
        "total_papers": int(len(df)),
        "date_range": f"{int(df['year'].min())} - {int(df['year'].max())}",
        "categories": df["tech_domain"].value_counts().to_dict(),
        "papers_per_year": df["year"].value_counts().sort_index().to_dict(),
        "collection_date": timestamp,
    }

    stats_path = out_dir / f"{filename_prefix}_{timestamp}_stats.json"
    stats_path.write_text(json.dumps(stats, indent=2))
    logger.info("Saved stats: {}", stats_path)

    return csv_path, json_path, stats_path


In [12]:
def quick_sample(
    n_per_category: int = 50,
    years: Iterable[int] | None = None,
) -> pd.DataFrame:
    """Quick sample collection for testing the pipeline."""
    years = list(years or [2015, 2018, 2021, 2024])
    logger.info(
        "Quick sample: {} papers × {} categories × {} years",
        n_per_category,
        len(TECH_CATEGORIES),
        len(years),
    )

    all_papers: List[Dict] = []

    for cat_code, cat_name in TECH_CATEGORIES.items():
        for year in years:
            logger.info("Sampling {} - {}", cat_name, year)
            papers = fetch_papers_for_category_year(cat_code, year, n_per_category)
            all_papers.extend(papers)
            time.sleep(0.5)

    df = pd.DataFrame(all_papers)
    if not df.empty:
        df = df.drop_duplicates(subset=["arxiv_id"], keep="first")

    return df


In [13]:
def collect_specific_domains(
    domains: Iterable[str],
    start_year: int = 2015,
    end_year: int = 2024,
    papers_per_year: int = 200,
) -> pd.DataFrame:
    """Collect data for specific tech domains only."""
    domains_set = set(domains)
    selected_cats = {
        code: name for code, name in TECH_CATEGORIES.items() if name in domains_set
    }

    if not selected_cats:
        logger.error(
            "No matching domains found. Requested: {}. Available: {}",
            sorted(domains_set),
            sorted(set(TECH_CATEGORIES.values())),
        )
        return pd.DataFrame()

    return collect_all_data(
        categories=selected_cats,
        start_year=start_year,
        end_year=end_year,
        papers_per_cat_year=papers_per_year,
    )

In [15]:
# quick driver cell (instead of a clean __main__ guard)
# tweak these by hand when running in the notebook
mode = "quick"  # options: "quick", "full", "domains"
selected_domains = ["Machine Learning", "Computer Vision"]

if mode == "quick":
    df_result = quick_sample(n_per_category=30)
elif mode == "full":
    df_result = collect_all_data()
elif mode == "domains":
    df_result = collect_specific_domains(selected_domains, start_year=2015, end_year=2024)
else:
    df_result = pd.DataFrame()

print("scraped rows:", len(df_result))


2026-02-24 11:48:52.286 | INFO     | __main__:quick_sample:7 - Quick sample: 30 papers × 22 categories × 4 years
2026-02-24 11:48:52.287 | INFO     | __main__:quick_sample:18 - Sampling Artificial Intelligence - 2015
2026-02-24 11:48:52.924 | INFO     | __main__:quick_sample:18 - Sampling Artificial Intelligence - 2018
2026-02-24 11:48:53.549 | INFO     | __main__:quick_sample:18 - Sampling Artificial Intelligence - 2021
2026-02-24 11:48:54.196 | INFO     | __main__:quick_sample:18 - Sampling Artificial Intelligence - 2024
2026-02-24 11:48:54.959 | INFO     | __main__:quick_sample:18 - Sampling Machine Learning - 2015
2026-02-24 11:48:55.664 | INFO     | __main__:quick_sample:18 - Sampling Machine Learning - 2018
2026-02-24 11:48:56.309 | INFO     | __main__:quick_sample:18 - Sampling Machine Learning - 2021
2026-02-24 11:48:57.049 | INFO     | __main__:quick_sample:18 - Sampling Machine Learning - 2024
2026-02-24 11:48:57.710 | INFO     | __main__:quick_sample:18 - Sampling Neural Net

scraped rows: 2161
